# Curate a microviridin-focused MdnC set (BLAST-window)

**Why this notebook exists:** the ConSurf auto-set was too *broad* (median ~28% identity, mostly distant Actinobacteria ATP-grasp enzymes), which washed out the microviridin-specific α7 leader cluster. This rebuilds the homolog set as a **Goldilocks set** — sequences 35–90% identical to *your* MdnC, macrocyclase-sized, redundancy-removed — so microviridin-specific residues can actually surface.

**What changed from your Week 2 notebook (3 things):**
1. **Band-pass filter** — instead of sampling across all identities, keep only the **35–90%** band. Below 35% is the alignment "twilight zone"; above 90% is redundant.
2. **Macrocyclase screen** — keep only sequences of the right length (~280–360 aa) that align over most of the query, so you get real full-domain ATP-grasp ligases, not fragments or fusions.
3. **Redundancy removal** — drop sequences that are >90% identical *to each other* (a CD-HIT stand-in, no install needed), so a cluster of near-identical strains can't inflate conservation.

Output: `data/mdnC_curated_aligned.fasta` — the MSA you upload to ConSurf as a **custom MSA**.


### Setup

In [2]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 19.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [biopython]/2 [biopython]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import time, shutil, statistics, subprocess
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import Entrez, SeqIO
from Bio.Seq import Seq

Entrez.email = "phamminh1@ufl.edu"   # <-- your real email
assert "YOUR_EMAIL" not in Entrez.email, "Put your real email above first."
DATA = Path("../data") if Path("../data").exists() else Path("data"); DATA.mkdir(exist_ok=True)
print("data folder:", DATA.resolve())

data folder: /Users/phamchutuanminh/Documents/microviridin-project/data


### Load the MdnC query from 5IG9 (longest chain)

In [4]:
matches  = [p for p in DATA.glob("*.fasta") if "5ig9" in p.name.lower()]
matches += [p for p in DATA.glob("*.txt")   if "5ig9" in p.name.lower()]
if not matches: raise FileNotFoundError(f"Put the 5IG9 FASTA in {DATA.resolve()}")
query_record = max(SeqIO.parse(matches[0], "fasta"), key=lambda r: len(r.seq))
query_seq = str(query_record.seq); QLEN=len(query_seq)
print(f"MdnC query: {QLEN} residues")

MdnC query: 333 residues


### BLAST — reuse your existing wide BLAST if you have it (else run it once)

In [5]:
blast_xml = None
for name in ("mdnC_blast_wide.xml","mdnC_blast.xml"):
    if (DATA/name).exists(): blast_xml = DATA/name; break

if blast_xml:
    print("Reusing saved BLAST:", blast_xml.name, "(no re-run needed)")
else:
    blast_xml = DATA/"mdnC_blast_wide.xml"
    print("No saved BLAST found. Running a wide BLAST (up to 1000 hits) ~10-20 min...")
    h = NCBIWWW.qblast("blastp","nr",query_seq,hitlist_size=1000)
    blast_xml.write_text(h.read()); h.close(); print("saved", blast_xml.name)

Reusing saved BLAST: mdnC_blast_wide.xml (no re-run needed)


### Step 1 — the band-pass filter (the key change)
We score each hit by **% of the whole query that is identical** (stricter than BLAST's local identity — this is what stops distant enzymes from sneaking in the way they did in ConSurf), and keep the **35–90%** band with good coverage.

In [6]:
LOW, HIGH, MIN_COV = 35.0, 90.0, 70.0

band=[]; seen=set()
with open(blast_xml) as fh:
    for al in NCBIXML.read(fh).alignments:
        b = al.hsps[0]
        pid = 100.0*b.identities  / QLEN     # % of the FULL query that matches identically
        cov = 100.0*b.align_length / QLEN     # how much of the query the hit spans
        if LOW <= pid <= HIGH and cov >= MIN_COV and al.accession not in seen:
            seen.add(al.accession); band.append((al.accession, pid))

band.sort(key=lambda x:-x[1])
print(f"{len(band)} hits fall in the {LOW:.0f}-{HIGH:.0f}% band with >={MIN_COV:.0f}% coverage")
if band:
    ps=[p for _,p in band]; print(f"  identity-to-query spans {min(ps):.0f}-{max(ps):.0f}%")
accessions=[a for a,_ in band]

875 hits fall in the 35-90% band with >=70% coverage
  identity-to-query spans 42-86%


/opt/homebrew/Cellar/jupyterlab/4.6.1/libexec/lib/python3.14/site-packages/Bio/Blast/NCBIXML.py:977: BiopythonParserWarning: NCBIXML: Ignored: '\nCREATE_VIEW\n\n\n'
  warnings.warn(


### Step 2 — fetch full sequences + macrocyclase length screen

In [ ]:
MINLEN, MAXLEN = 280, 360     # MdnB/MdnC are ~325 aa
out_raw = DATA/"mdnC_curated_raw.fasta"
collected=[]; seen_seqs={query_seq}
for i in range(0,len(accessions),50):
    h=Entrez.efetch(db="protein", id=",".join(accessions[i:i+50]), rettype="fasta", retmode="text")
    for r in SeqIO.parse(h,"fasta"):
        s=str(r.seq)
        if MINLEN<=len(s)<=MAXLEN and s not in seen_seqs:
            seen_seqs.add(s); collected.append(r)
    h.close(); time.sleep(0.5)
query_record.id="MdnC_5IG9_query"; query_record.description="MdnC query"
SeqIO.write([query_record]+collected, out_raw, "fasta")
print(f"{len(collected)} macrocyclase-sized sequences kept ({MINLEN}-{MAXLEN} aa) + query")

### Step 3 — align (MAFFT)

In [ ]:
raw_aln = DATA/"mdnC_curated_raw_aligned.fasta"
if shutil.which("mafft"):
    with open(raw_aln,"w") as o:
        subprocess.run(["mafft","--auto",str(out_raw)], stdout=o, check=True)
    print("aligned ->", raw_aln.name)
else:
    print("Install MAFFT first:  brew install mafft")

### Step 4 — redundancy removal (CD-HIT stand-in)
Two homologs can each be ~85% to the query but ~98% to *each other* — those inflate conservation. We keep one representative per >90%-identical cluster, then trim any all-gap columns.

In [ ]:
REDUN = 90.0
aln = list(SeqIO.parse(raw_aln, "fasta"))
def pid(a,b):
    s=t=0
    for x,y in zip(a,b):
        if x!="-" and y!="-": t+=1; s+=(x==y)
    return 100*s/t if t else 0

kept=[]
for rec in aln:                                   # query is first -> always kept
    if all(pid(str(rec.seq),str(k.seq)) < REDUN for k in kept):
        kept.append(rec)

cols=[i for i in range(len(kept[0].seq)) if any(r.seq[i] != "-" for r in kept)]
for r in kept: r.seq = Seq("".join(r.seq[i] for i in cols))
final = DATA/"mdnC_curated_aligned.fasta"
SeqIO.write(kept, final, "fasta")

q=str(kept[0].seq); ids=sorted(pid(q,str(r.seq)) for r in kept[1:])
print(f"Final curated set: {len(kept)} sequences (query + {len(kept)-1} homologs)  [from {len(aln)} pre-prune]")
if ids: print(f"identity-to-query: min {min(ids):.0f} | median {statistics.median(ids):.0f} | max {max(ids):.0f}")
if len(kept) < 50:
    print("\n<50 sequences is thin for ConSurf. Options: lower the 35 floor to ~30,")
    print("loosen the length screen, or enrich with the RODEO graspetide set next.")

## Next
Upload **`data/mdnC_curated_aligned.fasta`** to ConSurf as a **custom MSA**, with **PDB 5IG9, chain A** as the structure (your query sequence in the MSA matches that chain).

**Success check:** does the **α7 cluster (E191 / D192 / N195)** now rise toward conserved (grade ≥6, expected as acidic E/D), while the catalytic core and DxR R243 stay high? If yes, the focused set fixed the signal and you can read off new microviridin-specific positions.

Then commit: *"MdnC curated 35-90% microviridin set + alignment."*
